# Cruzamento sisgi × CNPJ

**Objetivo:** pegar os códigos de lotação da aba `sisgi` (base em uso) e buscar o CNPJ correspondente na aba `cnpj` (lista geral), padronizando o formato do CNPJ.

**Fluxo:**
1. Configuração
2. Funções auxiliares
3. Leitura das abas
4. Cruzamento (o "PROCV" do pandas)
5. Padronização do CNPJ
6. Status de cada linha
7. DataFrame final
8. Conferência
9. Exportação (só execute quando estiver tudo certo)

> O arquivo original **não é alterado**. O tratamento é todo feito em DataFrame; a exportação só grava um arquivo novo.

## 1) Configuração
Ajuste aqui os nomes do arquivo, das abas e das colunas. O restante do notebook não precisa mudar.

In [1]:
import re
import pandas as pd

ARQUIVO_ENTRADA = "lotacoes.xlsx"   # coloque o arquivo na mesma pasta do notebook

ABA_BASE = "sisgi"        # sua base (só os códigos que você usa)
COL_BASE = "id_lotacao"

ABA_CNPJ = "cnpj"         # lista geral (muitas escolas que você não usa)
COL_COD  = "Cod_Lotação"
COL_CNPJ = "CNPJ"

## 2) Funções auxiliares
- `so_digitos`: deixa só os números e completa zeros à esquerda até 14 dígitos. Resolve CNPJ que virou número e perdeu o zero inicial (`3143474000180`), com espaço (`03158670/0001 -29`) ou com formato incompleto (`04560246/0001-79`).
- `mascara`: aplica o formato `00.000.000/0000-00`.
- `cnpj_valido`: confere os 2 dígitos verificadores, para achar CNPJ digitado errado.

In [ ]:
def so_digitos(valor, tamanho=14):
    """Mantém só os números e completa com zeros à esquerda. Vazio -> None."""
    if pd.isna(valor):
        return None
    numeros = re.sub(r"\D", "", str(valor))      # \D = tudo que NÃO é dígito
    return numeros.zfill(tamanho) if numeros else None


def mascara(digitos):
    """03158670000129 -> 03.158.670/0001-29"""
    if pd.isna(digitos):                         # o pandas troca None por NaN
        return None
    d = digitos
    return f"{d[:2]}.{d[2:5]}.{d[5:8]}/{d[8:12]}-{d[12:]}"


def cnpj_valido(digitos):
    """Confere os dois dígitos verificadores do CNPJ."""
    if pd.isna(digitos) or len(digitos) != 14 or len(set(digitos)) == 1:
        return False

    def dv(base, pesos):
        resto = sum(int(n) * p for n, p in zip(base, pesos)) % 11
        return "0" if resto < 2 else str(11 - resto)

    pesos1 = [5, 4, 3, 2, 9, 8, 7, 6, 5, 4, 3, 2]
    pesos2 = [6] + pesos1
    return digitos[12] == dv(digitos[:12], pesos1) and digitos[13] == dv(digitos[:13], pesos2)

## 3) Leitura das abas
O CNPJ é lido como texto (`dtype=object`) para o pandas não converter em número e apagar zeros à esquerda.

In [ ]:
df_sisgi = pd.read_excel(ARQUIVO_ENTRADA, sheet_name=ABA_BASE)
df_cnpj  = pd.read_excel(ARQUIVO_ENTRADA, sheet_name=ABA_CNPJ, dtype={COL_CNPJ: object})

print(f"sisgi: {len(df_sisgi)} linhas | cnpj: {len(df_cnpj)} linhas")
df_sisgi.head()

In [ ]:
df_cnpj.head()

## 4) Cruzamento
`merge(how="left")` é o equivalente do PROCV: mantém **todas** as linhas do `sisgi` e traz o CNPJ quando o código existir na aba `cnpj`. As escolas que você não usa ficam de fora.

`indicator=True` cria a coluna `_merge`, que diz se o código foi encontrado (`both`) ou não (`left_only`).

In [ ]:
df = df_sisgi.merge(
    df_cnpj[[COL_COD, COL_CNPJ]],
    left_on=COL_BASE,
    right_on=COL_COD,
    how="left",
    indicator=True,
)

df["_merge"].value_counts()

## 5) Padronização do CNPJ

In [ ]:
df["CNPJ_original"]    = df[COL_CNPJ]
df["CNPJ_14_digitos"]  = df[COL_CNPJ].map(so_digitos)
df["CNPJ_padronizado"] = df["CNPJ_14_digitos"].map(mascara)

# Mostra só os CNPJs que estavam fora do padrão e como ficaram
fora_padrao = df[
    df["CNPJ_original"].notna()
    & (df["CNPJ_original"].astype(str) != df["CNPJ_padronizado"])
]
fora_padrao[[COL_BASE, "CNPJ_original", "CNPJ_padronizado"]]

## 6) Status de cada linha
Cada lotação recebe um diagnóstico: `OK`, `Lotação não existe na aba cnpj`, `Sem CNPJ na aba cnpj` ou `CNPJ com dígito verificador inválido`.

In [ ]:
def status(linha):
    if linha["_merge"] == "left_only":
        return "Lotação não existe na aba cnpj"
    if pd.isna(linha["CNPJ_14_digitos"]):
        return "Sem CNPJ na aba cnpj"
    if not cnpj_valido(linha["CNPJ_14_digitos"]):
        return "CNPJ com dígito verificador inválido"
    return "OK"

df["status"] = df.apply(status, axis=1)
df["status"].value_counts()

## 7) DataFrame final
Só as colunas que interessam.

In [ ]:
df_final = df[
    [COL_BASE, "CNPJ_original", "CNPJ_14_digitos", "CNPJ_padronizado", "status"]
].copy()

df_final.head(10)

## 8) Conferência
Linhas que merecem atenção (qualquer status diferente de `OK`) e duplicidades.

In [ ]:
problemas = df_final[df_final["status"] != "OK"]
print(f"Total: {len(df_final)} | OK: {(df_final['status'] == 'OK').sum()} | Com problema: {len(problemas)}")
problemas

In [ ]:
# O mesmo CNPJ aparecendo em mais de uma lotação (pode ser normal, só confira)
repetidos = df_final[df_final["CNPJ_14_digitos"].notna() & df_final["CNPJ_14_digitos"].duplicated(keep=False)]
print(f"Lotações com CNPJ repetido: {len(repetidos)}")
repetidos.sort_values("CNPJ_14_digitos")

## 9) Exportação
Só rode esta célula quando o `df_final` estiver como você quer. Ela cria um arquivo **novo**; o `lotacoes.xlsx` não é alterado.

Para exportar apenas o código e o CNPJ, troque `df_final` por `df_final[[COL_BASE, "CNPJ_padronizado"]]`.

In [ ]:
ARQUIVO_SAIDA = "sisgi_cnpj.xlsx"

df_final.to_excel(ARQUIVO_SAIDA, sheet_name="sisgi_cnpj", index=False)
print(f"Arquivo gerado: {ARQUIVO_SAIDA}")